# Этап 5 — MLP

## 1. Настройка окружения

In [1]:
# Базовые библиотеки для работы с JSON, путями, таблицами и PyTorch.
import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# Добавляем папку src, чтобы импортировать общие функции метрик проекта.
sys.path.append(str(Path.cwd().parent / "src"))

from utils import compute_metrics, find_best_threshold_f2

# Основные настройки эксперимента и пути к данным/артефактам.
SEED = 42
PROCESSED_DIR = Path("../data/processed")
RESULTS_DIR = Path("../results")
METRICS_DIR = RESULTS_DIR / "metrics"
PREDICTIONS_DIR = RESULTS_DIR / "predictions"

# Папки создаются заранее, чтобы сохранение результатов не упало в конце ноутбука.
METRICS_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

# Фиксируем seed для воспроизводимости инициализации и перемешивания батчей.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Выбираем устройство: CUDA приоритетнее, затем MPS, иначе CPU.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Устройство: {DEVICE}")

Устройство: cuda


## 2. Загрузка подготовленных данных

На этом шаге загружаются уже подготовленные `train`, `val` и `test` split из `data/processed`.

Отдельно читается `feature_types.json`, где сохранены списки числовых и категориальных признаков. Это важно, потому что MLP обрабатывает эти два типа признаков по-разному: числовые признаки нормализуются, а категориальные проходят через embedding-слои.


In [2]:
# Загружаем готовые split, полученные на этапе preprocessing.
train = pd.read_parquet(PROCESSED_DIR / "train.parquet")
val   = pd.read_parquet(PROCESSED_DIR / "val.parquet")
test  = pd.read_parquet(PROCESSED_DIR / "test.parquet")

# feature_types хранит разделение признаков на числовые и категориальные.
with open(PROCESSED_DIR / "feature_types.json", encoding="utf-8") as f:
    feature_types = json.load(f)

num_cols = feature_types["numeric"]
cat_cols = feature_types["categorical"]

# Быстрая проверка размеров и дисбаланса классов.
print(f"train: {train.shape}  val: {val.shape}  test: {test.shape}")
print(f"Числовые: {len(num_cols)}  Категориальные: {len(cat_cols)}")
print(f"Доля положительного класса в train: {train['target'].mean():.4f}")

train: (41982, 26)  val: (13994, 26)  test: (13994, 26)
Числовые: 12  Категориальные: 13
Доля положительного класса в train: 0.0897


## 3. Выделение целевой переменной

На этом шаге из каждого split отдельно выделяется столбец `target`. Он переводится в `np.float32`, потому что дальше используется бинарная функция потерь `BCEWithLogitsLoss`, которая ожидает вещественные целевые значения.

In [3]:
# Целевая переменная нужна отдельно от признаков.
# float32 подходит для BCEWithLogitsLoss в бинарной классификации.
y_train = train["target"].values.astype(np.float32)
y_val   = val["target"].values.astype(np.float32)
y_test  = test["target"].values.astype(np.float32)

## 4. Кодирование категорий

PyTorch embedding-слои работают не со строками, а с целочисленными индексами. Поэтому для каждого категориального признака строится словарь `значение -> индекс`, причём словарь обучается только на train.

Индекс `0` оставляется для неизвестных категорий, которые могут встретиться в validation или test.


In [4]:
# Маппинг строка -> индекс строится только по train, чтобы не было утечки из val/test.
# Индекс 0 зарезервирован для неизвестных категорий в validation/test.
cat_vocabs: dict[str, dict] = {}
for c in cat_cols:
    vals = train[c].unique().tolist()
    cat_vocabs[c] = {v: i + 1 for i, v in enumerate(vals)}

# Размер словаря + 1, потому что индекс 0 занят под неизвестные значения.
cat_dims = {c: len(cat_vocabs[c]) + 1 for c in cat_cols}


def encode_cats(df: pd.DataFrame, vocabs: dict, cols: list) -> np.ndarray:
    """Заменяет категориальные значения их индексами для embedding-слоёв."""
    out = np.zeros((len(df), len(cols)), dtype=np.int64)
    for j, c in enumerate(cols):
        vocab = vocabs[c]
        # Если категории не было в train, ставим 0.
        out[:, j] = [vocab.get(v, 0) for v in df[c]]
    return out


# Получаем матрицы индексов категорий для каждого split.
Xc_train = encode_cats(train, cat_vocabs, cat_cols)
Xc_val   = encode_cats(val,   cat_vocabs, cat_cols)
Xc_test  = encode_cats(test,  cat_vocabs, cat_cols)

print("Категориальные признаки закодированы — train:", Xc_train.shape, " val:", Xc_val.shape, " test:", Xc_test.shape)

Категориальные признаки закодированы — train: (41982, 13)  val: (13994, 13)  test: (13994, 13)


## 5. Нормализация числовых признаков

Числовые признаки переводятся к сопоставимому масштабу через `StandardScaler`. Если признаки имеют сильно разные диапазоны, оптимизатору сложнее стабильно обновлять веса. Scaler обучается только на train, а затем применяется к validation и test.


In [5]:
# StandardScaler обучается только на train.
scaler = StandardScaler()

# Числовые признаки приводятся к масштабу со средним 0 и стандартным отклонением 1.
Xn_train = scaler.fit_transform(train[num_cols].values).astype(np.float32)

# Для val/test используем параметры scaler из train, чтобы не было утечки данных.
Xn_val   = scaler.transform(val[num_cols].values).astype(np.float32)
Xn_test  = scaler.transform(test[num_cols].values).astype(np.float32)

print("Числовые признаки нормализованы — train:", Xn_train.shape, " val:", Xn_val.shape, " test:", Xn_test.shape)

Числовые признаки нормализованы — train: (41982, 12)  val: (13994, 12)  test: (13994, 12)


## 6. Подготовка тензоров и DataLoader

После кодирования и нормализации данные переводятся в PyTorch-тензоры нужных типов. Категориальные признаки становятся `long`, потому что embedding-слои принимают индексы, а числовые признаки и целевая переменная становятся `float32`.

Для train создаётся `TensorDataset` и `DataLoader` с `batch_size=256`. DataLoader перемешивает обучающие примеры и выдаёт batch, что делает обучение быстрее и стабильнее, чем обновление весов по всей выборке сразу.


In [6]:
def to_tensors(Xc, Xn, y):
    """Переводит numpy-массивы в типы, которые ожидает PyTorch-модель."""
    return (
        # Категории идут в nn.Embedding, поэтому нужен целочисленный long.
        torch.tensor(Xc, dtype=torch.long),
        # Числовые признаки и target участвуют в вычислениях loss, поэтому float32.
        torch.tensor(Xn, dtype=torch.float32),
        torch.tensor(y,  dtype=torch.float32),
    )


# Отдельно храним категориальные признаки, числовые признаки и target.
tc, tn, ty = to_tensors(Xc_train, Xn_train, y_train)
vc, vn, vy = to_tensors(Xc_val,   Xn_val,   y_val)
ec, en, ey = to_tensors(Xc_test,  Xn_test,  y_test)

# DataLoader отдаёт train mini-batch-ами и перемешивает их каждую эпоху.
train_loader = DataLoader(
    TensorDataset(tc, tn, ty),
    batch_size=256,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)

print(f"DataLoader: {len(train_loader)} батчей на эпоху  (batch_size=256)")

DataLoader: 164 батчей на эпоху  (batch_size=256)


## 7. Архитектура MLP

Модель состоит из двух частей: обработки категориальных признаков через embeddings и обычной полносвязной MLP-головы. Для каждого категориального признака создаётся отдельный embedding-слой, который учит компактное числовое представление категорий.

Дальше все embedding-векторы объединяются с нормализованными числовыми признаками в один общий вектор. Этот вектор проходит через слои `Linear`, `ReLU` и `Dropout`, а последний слой возвращает один logit для бинарной классификации.

На выходе не используется `sigmoid` внутри модели, потому что loss `BCEWithLogitsLoss` сам совмещает sigmoid и бинарную кросс-энтропию в более устойчивой численной форме.


In [7]:
class MLP(nn.Module):
    def __init__(
        self,
        cat_dims: list[int],
        emb_dims: list[int],
        n_num: int,
        hidden: tuple[int, ...] = (256, 128),
        dropout: float = 0.3,
    ):
        super().__init__()

        # Для каждого категориального признака создаётся свой embedding-слой.
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_emb, emb_dim)
            for num_emb, emb_dim in zip(cat_dims, emb_dims)
        ])

        # На вход MLP идут все embedding-векторы и нормализованные числовые признаки.
        in_dim = sum(emb_dims) + n_num
        layers: list[nn.Module] = []

        # Полносвязная голова постепенно сжимает общий вектор признаков.
        for h in hidden:
            layers += [
                nn.Linear(in_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            in_dim = h

        # Один выходной logit нужен для бинарной классификации.
        layers.append(nn.Linear(in_dim, 1))

        self.head = nn.Sequential(*layers)

    def forward(self, x_cat: torch.Tensor, x_num: torch.Tensor) -> torch.Tensor:
        # Достаём embedding-вектор для каждого категориального признака.
        embs = [emb(x_cat[:, j]) for j, emb in enumerate(self.embeddings)]

        # Объединяем embeddings и числовые признаки в один вход модели.
        x = torch.cat(embs + [x_num], dim=1)
        return self.head(x).squeeze(1)

## 8. Инициализация модели и учёт дисбаланса

Здесь рассчитываются размеры embedding-слоёв, создаётся экземпляр MLP и выбирается оптимизатор `AdamW`. После создания модель переносится на выбранное устройство, чтобы обучение шло на GPU при наличии CUDA.

Так как положительный класс встречается заметно реже отрицательного, в `BCEWithLogitsLoss` добавляется `pos_weight=N_neg/N_pos`. Это увеличивает штраф за ошибки на положительном классе и помогает модели не свалиться в предсказание только большинства.

В конце выводятся размер модели, значение `pos_weight` и итоговая размерность входного вектора. Эти числа помогают быстро проверить, что признаки и embedding-слои собрались ожидаемым образом.


In [8]:
# Размер embedding зависит от числа категорий, но ограничен сверху 16.
EMB_DIMS     = [min(16, (d + 1) // 2) for d in cat_dims.values()]
CAT_DIMS_LST = list(cat_dims.values())

# Создаём модель и переносим её на выбранное устройство.
model = MLP(CAT_DIMS_LST, EMB_DIMS, n_num=len(num_cols)).to(DEVICE)

# pos_weight усиливает штраф за ошибки на редком положительном классе.
pos_weight = torch.tensor(
    [(y_train == 0).sum() / (y_train == 1).sum()], dtype=torch.float32
).to(DEVICE)

# BCEWithLogitsLoss принимает logits и сам применяет sigmoid внутри.
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Adam обновляет веса модели после каждого batch.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Проверочные числа помогают понять размерность модели.
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Параметров: {total_params:,}")
print(f"pos_weight : {pos_weight.item():.2f}")
print(f"Размер входа: {sum(EMB_DIMS) + len(num_cols)}")

Параметров: 62,375
pos_weight : 10.15
Размер входа: 102


## 9. Обучение с подбором порога

В этом блоке выполняется основной цикл обучения. На каждой эпохе модель проходит по train batch-ам, считает loss, делает backpropagation и обновляет веса через optimizer.

После эпохи модель оценивается на validation split. Вероятности превращаются в бинарные предсказания не по фиксированному порогу `0.5`, а по порогу, который максимизирует F2 на validation данных.

F2 выбран потому, что он сильнее учитывает recall, а в задаче реадмиссии пропуск пациента из группы риска обычно хуже лишнего ложного срабатывания. Лучшая версия весов сохраняется по validation F2, а `patience` останавливает обучение, если качество перестало улучшаться.


In [9]:
def predict_proba(model: nn.Module, xc: torch.Tensor, xn: torch.Tensor,
                  batch_size: int = 2048) -> np.ndarray:
    """Считает вероятности положительного класса без обучения модели."""
    model.eval()
    parts = []
    with torch.no_grad():
        for s in range(0, len(xc), batch_size):
            logits = model(xc[s:s+batch_size].to(DEVICE), xn[s:s+batch_size].to(DEVICE))
            parts.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(parts)


MAX_EPOCHS = 50
PATIENCE   = 7

# Храним лучшую модель по validation F2, а не просто последнюю эпоху.
best_val_f2   = -1.0
best_state    = None
patience_cnt  = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0

    # Обучение: forward -> loss -> backward -> optimizer step.
    for xc_b, xn_b, y_b in train_loader:
        xc_b, xn_b, y_b = xc_b.to(DEVICE), xn_b.to(DEVICE), y_b.to(DEVICE)

        optimizer.zero_grad()
        loss = criterion(model(xc_b, xn_b), y_b)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(y_b)

    # После эпохи оцениваем модель на validation split.
    val_proba = predict_proba(model, vc, vn)
    thr       = find_best_threshold_f2(y_val, val_proba)
    val_m     = compute_metrics(y_val, val_proba, thr)

    # Если F2 улучшился, сохраняем копию весов.
    improved = val_m["f2"] > best_val_f2
    if improved:
        best_val_f2  = val_m["f2"]
        best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_cnt = 0

    if epoch % 5 == 0 or improved:
        flag = " ← лучше" if improved else ""
        print(
            f"Эпоха {epoch:3d} | loss={epoch_loss/len(y_train):.4f}"
            f" | val_auc={val_m['roc_auc']:.4f}"
            f" | val_f2={val_m['f2']:.4f}{flag}"
        )

    # Early stopping: если долго нет улучшения, обучение завершается.
    if not improved:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f"Ранняя остановка на эпохе {epoch}  (patience={PATIENCE})")
            break

print(f"\nЛучший val F2: {best_val_f2:.4f}")

Эпоха   1 | loss=1.2302 | val_auc=0.6400 | val_f2=0.3593 ← лучше
Эпоха   2 | loss=1.1932 | val_auc=0.6485 | val_f2=0.3642 ← лучше
Эпоха   4 | loss=1.1704 | val_auc=0.6540 | val_f2=0.3662 ← лучше
Эпоха   5 | loss=1.1683 | val_auc=0.6476 | val_f2=0.3650
Эпоха   8 | loss=1.1454 | val_auc=0.6537 | val_f2=0.3708 ← лучше
Эпоха  10 | loss=1.1277 | val_auc=0.6505 | val_f2=0.3656
Эпоха  15 | loss=1.0823 | val_auc=0.6582 | val_f2=0.3702
Ранняя остановка на эпохе 15  (patience=7)

Лучший val F2: 0.3708


## 10. Предсказания лучшей модели

После обучения восстанавливаются веса модели с лучшим validation F2, а не просто веса последней эпохи. Это нужно, потому что ближе к концу обучения модель могла начать переобучаться или случайно дать худшее качество на validation.

Затем считаются вероятности положительного класса для train, validation и test. Эти вероятности нужны и для расчёта метрик, и для сохранения предсказаний, которые затем используются в сравнительных ноутбуках.


In [10]:
# Возвращаем веса лучшей эпохи перед финальным расчётом метрик.
model.load_state_dict(best_state)

# Считаем вероятности для всех split.
train_proba = predict_proba(model, tc, tn)
val_proba   = predict_proba(model, vc, vn)
test_proba  = predict_proba(model, ec, en)

print(f"Размеры вероятностей — train: {train_proba.shape}  val: {val_proba.shape}  test: {test_proba.shape}")

Размеры вероятностей — train: (41982,)  val: (13994,)  test: (13994,)


## 11. Итоговые метрики

Финальный порог выбирается только на validation split по максимуму F2. После выбора он фиксируется и без дополнительной подстройки применяется к train, validation и test.

Для каждого split считаются ROC-AUC, precision, recall и F2. ROC-AUC оценивает качество ранжирования вероятностей, а precision/recall/F2 показывают поведение уже после выбора конкретного порога классификации.

Особенно важно, что test split используется только для финальной оценки. Порог не подбирается по test, поэтому результат остаётся честной оценкой обобщающей способности модели.


In [11]:
# Порог фиксируется по val, применяется к test (нет утечки данных).
threshold = find_best_threshold_f2(y_val, val_proba)
print(f"Оптимальный порог (максимум val F2): {threshold:.4f}\n")

# Считаем одинаковый набор метрик для train/val/test.
split_results: dict[str, dict] = {}
for name, y_true, y_proba in [
    ("train", y_train, train_proba),
    ("val",   y_val,   val_proba),
    ("test",  y_test,  test_proba),
]:
    m = compute_metrics(y_true, y_proba, threshold)
    split_results[name] = m
    print(
        f"{name:5s} | roc_auc={m['roc_auc']:.4f}"
        f" | precision={m['precision']:.4f}"
        f" | recall={m['recall']:.4f}"
        f" | f2={m['f2']:.4f}"
    )

Оптимальный порог (максимум val F2): 0.4481

train | roc_auc=0.7465 | precision=0.1384 | recall=0.8370 | f2=0.4165
val   | roc_auc=0.6537 | precision=0.1235 | recall=0.7420 | f2=0.3708
test  | roc_auc=0.6507 | precision=0.1193 | recall=0.7155 | f2=0.3578


## 12. Сохранение артефактов

После оценки сохраняется JSON с метриками, выбранным порогом и основными гиперпараметрами модели. Такой файл нужен, чтобы сравнительные ноутбуки могли читать результаты автоматически, без ручного переноса чисел.

Также сохраняются CSV с вероятностями для validation и test. Они позволяют позже строить графики, сравнивать модели по одним и тем же объектам или подбирать общие пороги без повторного обучения MLP.


In [12]:
# Собираем метрики, порог и гиперпараметры в один JSON-артефакт.
artifact = {
    "metrics": split_results,
    "threshold": float(threshold),
    "hyperparams": {
        "seed": SEED,
        "hidden": [256, 128],
        "dropout": 0.3,
        "lr": 1e-3,
        "batch_size": 256,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "pos_weight": float(pos_weight.item()),
        "class_imbalance_method": "BCEWithLogitsLoss(pos_weight=N_neg/N_pos)",
    },
}

# Метрики нужны для сравнительных таблиц.
with open(METRICS_DIR / "mlp.json", "w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2)

# Вероятности сохраняются отдельно, чтобы строить графики и сравнивать модели без переобучения.
pd.DataFrame({"y_true": y_val,  "y_proba": val_proba}).to_csv(
    PREDICTIONS_DIR / "mlp_val.csv",  index=False)
pd.DataFrame({"y_true": y_test, "y_proba": test_proba}).to_csv(
    PREDICTIONS_DIR / "mlp_test.csv", index=False)

print("Сохранено:")
print(f"  results/metrics/mlp.json")
print(f"  results/predictions/mlp_val.csv   ({len(y_val)} строк)")
print(f"  results/predictions/mlp_test.csv  ({len(y_test)} строк)")

Сохранено:
  results/metrics/mlp.json
  results/predictions/mlp_val.csv   (13994 строк)
  results/predictions/mlp_test.csv  (13994 строк)


## 13. Вывод

MLP с embedding-слоями для категориальных признаков показал рабочее, но не лучшее качество для этой задачи. Модель хорошо смещена в сторону поиска положительного класса, поэтому recall высокий, но precision остаётся низким.

Метрики на test:

| Метрика | Значение |
|---|---:|
| ROC-AUC | 0.6507 |
| Precision | 0.1193 |
| Recall | 0.7155 |
| F2 | 0.3578 |

Порог классификации `0.4481` был выбран по максимуму F2 на validation и затем без дополнительной настройки применён к test.

Главный результат модели — высокий recall: она находит большую часть пациентов из положительного класса. Цена за это - много ложноположительных срабатываний, поэтому precision низкий. Для задачи риска реадмиссии такой компромисс допустим, потому что пропуск рискового пациента обычно хуже лишней проверки.

По сравнению с простой линейной моделью MLP добавляет нелинейности и обучаемые представления категорий, но для табличных медицинских данных этого всё ещё может быть недостаточно. Поэтому эту модель стоит рассматривать как промежуточную нейросетевую точку сравнения перед более сильными табличными подходами, такими как CatBoost, FT-Transformer и TabM.
